# Single-Query PubChem Retrieval

Paste a hand-entered spectrum (m/z + intensity peak list) below and get the
top-N most similar molecules from the full PubChem ICICLE-prediction HDF5.

Uses the same binning (0-750 Da, 1 Da bins) and cosine similarity as
`examples/scripts/evaluation/pubchem_global_retrieval.py`, but scoped to a
single ad-hoc query instead of the full NIST test set.

In [ ]:
import h5py
import numpy as np
import torch

HDF5_PATH = "../../results/inference/pubchem_predictions_rerun_260710.hdf5"
N_BINS, MIN_MZ, MAX_MZ = 750, 0, 750
CHUNK = 2_000_000
TOP_N = 10

## Enter the query spectrum
Paste m/z values and intensities (raw or %, doesn't matter — gets max-normalized).

In [ ]:
query_mz = np.array(
    [
        36.6,
        37.8,
        38.8,
        39.8,
        40.8,
        41.8,
        42.8,
        43.8,
        44.8,
        52.6,
        54.6,
        55.6,
        67.8,
        68.8,
        70.4,
        83.6,
        96.4,
        111.8,
    ]
)
query_intensity = np.array(
    [
        1024.0,
        2385.0,
        12475.0,
        26640.0,
        4787.0,
        1953.0,
        11368.0,
        1422.0,
        3285.0,
        570.0,
        1257.0,
        234.0,
        22952.0,
        3673.0,
        364.0,
        618.0,
        354.0,
        18488.0,
    ]
)

In [ ]:
def bin_spectrum(mz, intensity, min_mz=MIN_MZ, max_mz=MAX_MZ, n_bins=N_BINS):
    spec = np.zeros(n_bins, dtype=np.float32)
    bins = np.floor(mz - min_mz).astype(int)
    valid = (bins >= 0) & (bins < n_bins)
    np.add.at(spec, bins[valid], intensity[valid])
    spec /= spec.max()
    return spec


query_spec = bin_spectrum(query_mz, query_intensity)
query_t = torch.from_numpy(query_spec).unsqueeze(0)

## Optional: MW window pre-filter

A full scan of all ~93.6M PubChem rows takes several minutes (I/O-bound
reading the 110GB `intensities` array). Setting `MW_WINDOW_DA` narrows the
candidate pool to molecules within `+/- MW_WINDOW_DA` of a center mass
first, using the pre-sorted `sort_idx_mw`/`sorted_mw` index to build a
boolean mask over all rows (binary search, seconds), then only ranks cosine
similarity for masked-in rows. This mirrors `evaluate_mw_window` /
`_scan_window` in `pubchem_global_retrieval.py`.

Window center defaults to the highest observed m/z peak in the query
spectrum — the same MW proxy used there (real MW/precursor mass is usually
unknown for GC-MS spectra; the molecular ion peak is often weak/absent).

Set `MW_WINDOW_DA = None` to disable the filter and do the full scan.

**Important**: never fancy-index the HDF5 with a scattered row-index array
(`f["intensities"][row_indices]`) — h5py turns that into one I/O op per row
(or short run), which is catastrophically slower than reading each chunk
contiguously and masking in memory. This bit us directly: ~66k scattered
rows took minutes via fancy-indexing. Always build a boolean mask and read
chunks sequentially instead, as done below.

In [ ]:
MW_WINDOW_DA = 4  # +/- Da; set to None to scan the full database instead
MW_CENTER = float(query_mz.max())  # highest observed peak as MW proxy

print(
    f"MW window center: {MW_CENTER} Da, window: +/- {MW_WINDOW_DA} Da"
    if MW_WINDOW_DA is not None
    else "MW filter disabled — full scan"
)

## Scan PubChem predictions (chunked, streaming top-N)

ponytail: no GPU batching / async I/O, add if this needs to run often or on
a bigger file.

In [ ]:
best_sims = np.array([], dtype=np.float32)
best_idx = np.array([], dtype=np.int64)

with h5py.File(HDF5_PATH, "r") as f:
    n = f.attrs["n_molecules"]

    needed_mask = None
    if MW_WINDOW_DA is not None:
        sort_idx = f["sort_idx_mw"][:]
        sorted_mw = f["sorted_mw"][:]
        lo = np.searchsorted(sorted_mw, MW_CENTER - MW_WINDOW_DA)
        hi = np.searchsorted(sorted_mw, MW_CENTER + MW_WINDOW_DA, side="right")
        needed_mask = np.zeros(n, dtype=bool)
        needed_mask[sort_idx[lo:hi]] = True
        print(f"MW window matched {needed_mask.sum():,} candidates")

    for start in range(0, n, CHUNK):
        end = min(start + CHUNK, n)
        if needed_mask is not None and not needed_mask[start:end].any():
            continue

        cands_all = torch.from_numpy(f["intensities"][start:end])
        if needed_mask is not None:
            keep = needed_mask[start:end]
            keep_rows = np.where(keep)[0]
            cands = cands_all[keep_rows]
            rows = keep_rows + start
        else:
            cands = cands_all
            rows = np.arange(start, end)

        if len(rows) == 0:
            continue

        cand_norm = cands / cands.amax(dim=1, keepdim=True).clamp(min=1e-12)
        num = (cand_norm * query_t).sum(dim=1)
        den = cand_norm.norm(dim=1) * query_t.norm(dim=1)
        sims = (num / den.clamp(min=1e-12)).numpy()

        combined_sims = np.concatenate([best_sims, sims])
        combined_idx = np.concatenate([best_idx, rows])
        top = np.argsort(-combined_sims)[:TOP_N]
        best_sims = combined_sims[top]
        best_idx = combined_idx[top]

    sort_order_idx = np.sort(best_idx)
    order = np.argsort(best_idx)
    inv_order = np.argsort(order)
    smiles = np.array(
        [
            s.decode() if isinstance(s, bytes) else s
            for s in f["smiles"][sort_order_idx]
        ]
    )[inv_order]
    inchikey14 = np.array(
        [f["inchikey14"][i].decode() for i in sort_order_idx]
    )[inv_order]

In [ ]:
import pandas as pd

results = (
    pd.DataFrame(
        {
            "cosine_similarity": best_sims,
            "inchikey14": inchikey14,
            "smiles": smiles,
        }
    )
    .sort_values("cosine_similarity", ascending=False)
    .reset_index(drop=True)
)
results.index += 1
results.index.name = "rank"
results

## Mirror plots: query spectrum vs. each top-10 candidate

Each candidate's predicted spectrum comes straight from the PubChem HDF5
(`intensities` — already ICICLE-predicted, no re-inference needed here).
Query spectrum is the hand-entered one from above, binned the same way.

In [ ]:
from icicle.utils.visualization import set_style
from icicle.utils.visualization.mass_spectra import plot_mirrored_spectra

set_style("manuscript")

with h5py.File(HDF5_PATH, "r") as f:
    candidate_specs = f["intensities"][np.sort(best_idx)][inv_order]

mirror_figs = []
for rank, (sim, ik, sm, cand_spec) in enumerate(
    zip(best_sims, inchikey14, smiles, candidate_specs), 1
):
    fig = plot_mirrored_spectra(
        true_spec=query_spec,
        pred_spec=cand_spec / cand_spec.max(),
        true_smiles=None,
        pred_smiles=sm,
        true_label="Query (experimental)",
        predicted_label=f"PubChem #{rank} (ICICLE pred)",
        title=f"Rank {rank}  cosine={sim:.3f}  {ik}",
    )
    mirror_figs.append(fig)

## Fragment substructures for the top candidates

Re-runs ICICLE inference per-candidate (CPU/GPU, single molecule at a time)
to recover the fragment structures + atom/bond highlights — the stored
PubChem HDF5 only has final binned intensities, not fragment-level
attribution, so this step is required to draw substructures above peaks.

ponytail: re-inferring instead of trusting the HDF5 intensities to match
exactly — same checkpoint, so they should agree; if they don't, that's a
checkpoint-mismatch bug worth flagging, not silently ignored.

In [ ]:
CKPT = "checkpoints/entropy_random_s1/checkpoints/best-model-val_loss=0.1183-epoch=48.ckpt"
DEVICE = "cuda:0"
MAX_NODES = 100
THRESHOLD = 0.01
MAX_FRAGMENTS_PER_PLOT = 5
TOP_K_FOR_FRAGMENTS = 3  # re-inference is per-molecule; keep this small

from icicle.models.eims_predictor import EIMSPredictorFromFullEnumeration
from icicle.utils.visualization.mass_spectra import plot_mass_spectrum

eims_predictor = EIMSPredictorFromFullEnumeration(
    min_mz=MIN_MZ, max_mz=MAX_MZ, bin_width=1.0
)
eims_predictor.load_from_checkpoint(CKPT)
print("Model loaded.")

In [ ]:
fragment_figs = []
for rank, (sim, ik, sm) in enumerate(
    zip(
        best_sims[:TOP_K_FOR_FRAGMENTS],
        inchikey14[:TOP_K_FOR_FRAGMENTS],
        smiles[:TOP_K_FOR_FRAGMENTS],
    ),
    1,
):
    result = eims_predictor.predict_from_smiles(
        smiles=sm, max_nodes=MAX_NODES, threshold=THRESHOLD, device=DEVICE
    )
    fig = plot_mass_spectrum(
        mz_values=result["mz_bins"],
        intensities=result["intensities"],
        smiles=sm,
        fragments=result["fragments"],
        max_fragments=MAX_FRAGMENTS_PER_PLOT,
        title=f"Rank {rank}  cosine={sim:.3f}  {ik}",
    )
    fragment_figs.append(fig)

## Same retrieval against NEIMS-predicted PubChem

Reuses the identical MW-window + chunked-scan logic, pointed at NEIMS's own
full-PubChem prediction HDF5 instead of ICICLE's — same schema
(`intensities`, `sort_idx_mw`/`sorted_mw`, etc.), so no other changes needed.
Split matches the ICICLE checkpoint above (`entropy_random_s1` -> NEIMS
`random_s1`) for a fair head-to-head.

In [ ]:
NEIMS_HDF5_PATH = "../../baselines/neims/results/pubchem_predictions/neims_random_s1_pubchem_full.hdf5"


def scan_hdf5_topk(
    hdf5_path, query_t, mw_center, mw_window_da, chunk=CHUNK, top_n=TOP_N
):
    best_sims_ = np.array([], dtype=np.float32)
    best_idx_ = np.array([], dtype=np.int64)

    with h5py.File(hdf5_path, "r") as f:
        n = f["intensities"].shape[0]

        needed_mask = None
        if mw_window_da is not None:
            sort_idx = f["sort_idx_mw"][:]
            sorted_mw = f["sorted_mw"][:]
            lo = np.searchsorted(sorted_mw, mw_center - mw_window_da)
            hi = np.searchsorted(
                sorted_mw, mw_center + mw_window_da, side="right"
            )
            needed_mask = np.zeros(n, dtype=bool)
            needed_mask[sort_idx[lo:hi]] = True
            print(f"MW window matched {needed_mask.sum():,} candidates")

        for start in range(0, n, chunk):
            end = min(start + chunk, n)
            if needed_mask is not None and not needed_mask[start:end].any():
                continue

            cands_all = torch.from_numpy(f["intensities"][start:end])
            if needed_mask is not None:
                keep = needed_mask[start:end]
                keep_rows = np.where(keep)[0]
                cands = cands_all[keep_rows]
                rows = keep_rows + start
            else:
                cands = cands_all
                rows = np.arange(start, end)

            if len(rows) == 0:
                continue

            cand_norm = cands / cands.amax(dim=1, keepdim=True).clamp(
                min=1e-12
            )
            num = (cand_norm * query_t).sum(dim=1)
            den = cand_norm.norm(dim=1) * query_t.norm(dim=1)
            sims = (num / den.clamp(min=1e-12)).numpy()

            combined_sims = np.concatenate([best_sims_, sims])
            combined_idx = np.concatenate([best_idx_, rows])
            top = np.argsort(-combined_sims)[:top_n]
            best_sims_ = combined_sims[top]
            best_idx_ = combined_idx[top]

        sort_order_idx = np.sort(best_idx_)
        order = np.argsort(best_idx_)
        inv_order_ = np.argsort(order)
        smiles_ = np.array(
            [
                s.decode() if isinstance(s, bytes) else s
                for s in f["smiles"][sort_order_idx]
            ]
        )[inv_order_]
        inchikey14_ = np.array(
            [
                ik.decode() if isinstance(ik, bytes) else ik
                for ik in f["inchikey14"][sort_order_idx]
            ]
        )[inv_order_]

    return best_sims_, inchikey14_, smiles_


neims_sims, neims_inchikey14, neims_smiles = scan_hdf5_topk(
    NEIMS_HDF5_PATH, query_t, MW_CENTER, MW_WINDOW_DA
)

In [ ]:
neims_results = (
    pd.DataFrame(
        {
            "cosine_similarity": neims_sims,
            "inchikey14": neims_inchikey14,
            "smiles": neims_smiles,
        }
    )
    .sort_values("cosine_similarity", ascending=False)
    .reset_index(drop=True)
)
neims_results.index += 1
neims_results.index.name = "rank"
neims_results